# Softmax 与交叉熵：LogSumExp、梯度和 Mask 边界

**面试问题：为什么 Softmax 要减最大值，Cross Entropy 怎样从 Logits 稳定计算？**

## 回答主线

1. 直接对大 Logit 取指数会溢出，先减每行最大值不改变 Softmax 比例。
2. 交叉熵最好用 `logsumexp(logits)-target_logit`，避免先算极小概率再取 log。
3. 对单样本 Logit 的梯度是 `p-one_hot(y)`，可直接检查方向和总和为零。
4. Label Smoothing 改变目标分布，不是数值稳定技巧。
5. Mask 必须在归一化前应用；全 Mask 行没有合法概率分布，应显式拒绝或返回约定值。
6. 混合精度下仍应让归一化累加使用足够精度。

## 真实案例

五条客服文本要分到退款、物流、账号三个意图，Logit 包含 1000、-1200 等极值。我们先让朴素 `exp` 溢出，再手写稳定 Softmax、Cross Entropy 与梯度，逐样本打印概率和损失；最后复现注意力式全 Mask 行产生 NaN。使用可读的离线教学数据解释机制，指标不能外推为线上收益。

### 输入预览：五条文本与极端 Logits

In [1]:
import numpy as np  # 导入数组运算以手写 Softmax 和交叉熵。

classes = ["退款", "物流", "账号"]  # 定义三个客服意图。
samples = ["钱什么时候退", "包裹没有更新", "账号无法登录", "重复扣款", "快递明天到吗"]  # 构造五条可读文本。
logits = np.array([[1000.0, 998.0, 995.0], [-800.0, -790.0, -805.0], [-1200.0, -1210.0, -1188.0], [750.0, 740.0, 730.0], [2.0, 4.0, 1.0]], dtype=np.float64)  # 构造正负极端分数。
targets = np.array([0, 1, 2, 0, 1], dtype=int)  # 定义正确类别索引。
print("文本             target  logits")  # 输出输入表头。
for text, target, row in zip(samples, targets, logits):  # 逐样本展示业务语义和分数。
    print(f"{text:<15} {classes[target]:<4} {row}")  # 展示极端值不会改变相对排序。

文本             target  logits
钱什么时候退          退款   [1000.  998.  995.]
包裹没有更新          物流   [-800. -790. -805.]
账号无法登录          账号   [-1200. -1210. -1188.]
重复扣款            退款   [750. 740. 730.]
快递明天到吗          物流   [2. 4. 1.]


## Baseline 基线：直接 Exp 再归一化

In [2]:
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):  # 允许捕获溢出结果而不中断教学执行。
    naive_exponentials = np.exp(logits)  # 直接对 1000 和 750 取指数会产生 inf。
    naive_probabilities = naive_exponentials / naive_exponentials.sum(axis=1, keepdims=True)  # inf/inf 导致 NaN。
    naive_target_probabilities = naive_probabilities[np.arange(len(targets)), targets]  # 读取正确类概率。
    naive_losses = -np.log(naive_target_probabilities)  # 对零或 NaN 概率取对数。
print("朴素 exp：\n", naive_exponentials)  # 展示 inf 和下溢零。
print("朴素 probabilities：\n", naive_probabilities)  # 展示多个 NaN 行。
print("朴素 losses：", naive_losses)  # 展示数值不可用。
print(f"有限损失样本数={np.isfinite(naive_losses).sum()}/{len(samples)}")  # 量化基线失败。

朴素 exp：
 [[        inf         inf         inf]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [        inf         inf         inf]
 [ 7.3890561  54.59815003  2.71828183]]
朴素 probabilities：
 [[       nan        nan        nan]
 [       nan        nan        nan]
 [       nan        nan        nan]
 [       nan        nan        nan]
 [0.1141952  0.84379473 0.04201007]]
朴素 losses： [       nan        nan        nan        nan 0.16984602]
有限损失样本数=1/5


### 核心实现：稳定 Softmax、LogSumExp 与 CE

In [3]:
def stable_softmax(matrix):  # 手写按行稳定 Softmax。
    row_maximum = np.max(matrix, axis=1, keepdims=True)  # 找到每行最大 Logit。
    shifted = matrix - row_maximum  # 平移后最大值变为零且概率比例不变。
    exponentials = np.exp(shifted)  # 所有指数都位于 (0,1] 避免上溢。
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)  # 按行归一化。
    return probabilities, shifted, exponentials  # 返回概率和可观察中间量。

def logsumexp(matrix):  # 手写按行 LogSumExp。
    row_maximum = np.max(matrix, axis=1)  # 保存每行最大值用于平移和恢复。
    shifted_sum = np.exp(matrix - row_maximum[:, None]).sum(axis=1)  # 在稳定范围求指数和。
    return row_maximum + np.log(shifted_sum)  # 加回最大值获得原空间结果。

stable_probabilities, shifted_logits, stable_exponentials = stable_softmax(logits)  # 计算稳定概率。
stable_losses = logsumexp(logits) - logits[np.arange(len(targets)), targets]  # 直接从 Logit 计算正确类 NLL。
print("第一行平移 Logits：", shifted_logits[0], "指数：", np.round(stable_exponentials[0], 6))  # 展示减最大值过程。
print("样本  target  probabilities                  CE")  # 输出稳定结果表头。
for text, target, probabilities, loss in zip(samples, targets, stable_probabilities, stable_losses):  # 逐样本展示完整分布和损失。
    print(f"{text:<15} {classes[target]:<4} {np.round(probabilities, 6)} {loss:.6f}")  # 展示所有样本均有限。

第一行平移 Logits： [ 0. -2. -5.] 指数： [1.       0.135335 0.006738]
样本  target  probabilities                  CE
钱什么时候退          退款   [0.875601 0.1185   0.0059  ] 0.132845
包裹没有更新          物流   [4.50000e-05 9.99954e-01 0.00000e+00] 0.000046
账号无法登录          账号   [6.00000e-06 0.00000e+00 9.99994e-01] 0.000006
重复扣款            退款   [9.99955e-01 4.50000e-05 0.00000e+00] 0.000045
快递明天到吗          物流   [0.114195 0.843795 0.04201 ] 0.169846


## 结果解读：梯度为何是 p - one_hot

In [4]:
one_hot = np.zeros_like(stable_probabilities)  # 初始化目标分布矩阵。
one_hot[np.arange(len(targets)), targets] = 1.0  # 在正确类别位置写入一。
logit_gradients = stable_probabilities - one_hot  # 使用解析公式计算每个样本 CE 对 Logit 梯度。
print("样本  target  gradient                         行和")  # 输出梯度表头。
for text, target, gradient in zip(samples, targets, logit_gradients):  # 逐样本展示梯度方向。
    print(f"{text:<15} {classes[target]:<4} {np.round(gradient, 6)} {gradient.sum():+.2e}")  # 展示正确类梯度为负且行和为零。
epsilon = 1e-5  # 定义有限差分步长。
probe_logits = logits.copy()  # 复制 Logits 以检查第一样本正确类梯度。
probe_logits[0, 0] += epsilon  # 只扰动第一样本目标 Logit。
finite_difference = ((logsumexp(probe_logits)[0] - probe_logits[0, targets[0]]) - stable_losses[0]) / epsilon  # 计算数值梯度。
analytic_gradient = logit_gradients[0, 0]  # 读取解析梯度。
print(f"第一样本 target-logit 梯度：解析={analytic_gradient:.8f}，有限差分={finite_difference:.8f}")  # 验证推导。

样本  target  gradient                         行和
钱什么时候退          退款   [-0.124399  0.1185    0.0059  ] -3.47e-18
包裹没有更新          物流   [ 4.5e-05 -4.6e-05  0.0e+00] -1.13e-16
账号无法登录          账号   [ 6.e-06  0.e+00 -6.e-06] -3.51e-17
重复扣款            退款   [-4.5e-05  4.5e-05  0.0e+00] -9.71e-17
快递明天到吗          物流   [ 0.114195 -0.156205  0.04201 ] +0.00e+00
第一样本 target-logit 梯度：解析=-0.12439940，有限差分=-0.12439887


## 失败案例：一整行都被 Mask 时没有合法分布

In [5]:
masked_logits = np.array([[3.0, 1.0, -2.0], [-np.inf, -np.inf, -np.inf]], dtype=np.float64)  # 构造一行部分有效、一行全部无效的注意力式 Logit。
with np.errstate(invalid="ignore"):  # 捕获负无穷相减产生的 NaN。
    unsafe_masked_probabilities, _, _ = stable_softmax(masked_logits)  # 普通稳定 Softmax 仍无法定义全 Mask 行。

def masked_softmax(matrix, valid_mask):  # 实现带全 Mask 门禁的 Softmax。
    valid_counts = valid_mask.sum(axis=1)  # 统计每行合法位置数。
    if np.any(valid_counts == 0):  # 任一行没有合法候选。
        return None, [int(index) for index in np.where(valid_counts == 0)[0]]  # 返回明确错误行而不是 NaN。
    safe_logits = np.where(valid_mask, matrix, -np.inf)  # 在归一化前屏蔽无效位置。
    probabilities, _, _ = stable_softmax(safe_logits)  # 仅在至少一个合法位置时计算概率。
    return probabilities, []  # 返回概率和空错误列表。

valid_mask = np.array([[True, True, False], [False, False, False]])  # 定义第二行全 Mask。
safe_masked_probabilities, invalid_rows = masked_softmax(masked_logits, valid_mask)  # 执行安全门禁。
print("普通稳定 Softmax：\n", unsafe_masked_probabilities)  # 展示全 Mask 行仍为 NaN。
print("安全 Masked Softmax：", safe_masked_probabilities, "invalid_rows=", invalid_rows)  # 展示明确拒绝第二行。
print("生产边界：GPU Kernel 还需处理 FP16/BF16 累加、极长类别、融合 CE、ignore_index、label smoothing 和分布式词表切分。")  # 总结工程边界。

普通稳定 Softmax：
 [[0.8756006  0.11849965 0.00589975]
 [       nan        nan        nan]]
安全 Masked Softmax： None invalid_rows= [1]
生产边界：GPU Kernel 还需处理 FP16/BF16 累加、极长类别、融合 CE、ignore_index、label smoothing 和分布式词表切分。


## 回归测试：最后只保护概率、CE 梯度与全 Mask 门禁

In [6]:
assert np.isfinite(stable_losses).all() and np.allclose(stable_probabilities.sum(axis=1), 1.0)  # 验证所有极值样本损失有限且概率归一。
assert np.isfinite(naive_losses).sum() < len(samples)  # 验证朴素 Exp 失败探针确实存在。
assert np.allclose(logit_gradients.sum(axis=1), 0.0, atol=1e-12)  # 验证每行 Logit 梯度和为零。
assert abs(finite_difference - analytic_gradient) < 1e-5  # 验证解析梯度匹配有限差分。
assert np.isnan(unsafe_masked_probabilities[1]).all() and safe_masked_probabilities is None and invalid_rows == [1]  # 验证全 Mask NaN 与门禁修正。
print("回归测试通过：极值稳定性、概率归一、CE 梯度、有限差分和全 Mask 门禁均成立。")  # 用少量断言总结数值合同。

回归测试通过：极值稳定性、概率归一、CE 梯度、有限差分和全 Mask 门禁均成立。
